<a href="https://colab.research.google.com/github/adriansivina-droid/scraper-mef/blob/main/EXPORT/EXP_Selector_Partidas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ═══ BLOQUE 1 · CARGA DEL DICCIONARIO (ejecutar una vez por sesión) ═══
import os, io, time
import pandas as pd
import ipywidgets as w
from IPython.display import display

BASE = "https://raw.githubusercontent.com/adriansivina-droid/scraper-mef/main/EXPORT/arancel"
dic = pd.read_csv(f"{BASE}/diccionario_partidas.csv", dtype=str).fillna("")
corr = pd.read_csv(f"{BASE}/correlaciones_sunat.csv", dtype=str).fillna("")
VERS = ["2002", "2007", "2012", "2017", "2022"]
# Vigencia de cada versión del Arancel (fecha de entrada en vigor)
VIGOR = {"2002": (2002, 1), "2007": (2007, 4), "2012": (2012, 1), "2017": (2017, 1), "2022": (2022, 1)}
print(f"✅ Diccionario: {len(dic):,} subpartidas · {dic.CAPITULO.nunique()} capítulos · {len(corr):,} correlaciones")


In [ ]:
# ═══ BLOQUE 2 · SELECTORES (elige y luego ejecuta el Bloque 3) ═══
from datetime import date
_años = list(range(date.today().year, 2001, -1))
_ancho = {"description_width": "120px"}
_caps = (dic[["CAPITULO", "CAPITULO_DESC"]].drop_duplicates().sort_values("CAPITULO")
         .apply(lambda r: (f"{r.CAPITULO} · {r.CAPITULO_DESC}"[:90], r.CAPITULO), axis=1).tolist())

sel_anio_ini = w.Dropdown(options=_años, value=2005, description="Año inicio", style=_ancho)
sel_anio_fin = w.Dropdown(options=_años, value=_años[0], description="Año fin", style=_ancho)
sel_caps = w.SelectMultiple(options=_caps, value=(), rows=12, description="Capítulos",
                            style=_ancho, layout=w.Layout(width="760px"))
txt_buscar = w.Text(value="", description="Buscar texto", placeholder="ej. madera, café, 4412 (opcional)",
                    style=_ancho, layout=w.Layout(width="500px"))
chk_parciales = w.Checkbox(value=True, description="Incluir partidas parciales (con equivalencias)")
chk_drive = w.Checkbox(value=True, description="Guardar el resultado en Drive")
display(w.VBox([w.HBox([sel_anio_ini, sel_anio_fin]), sel_caps,
                w.HTML("<small>Sin capítulos elegidos = todos. Ctrl/Cmd + clic para varios.</small>"),
                txt_buscar, chk_parciales, chk_drive]))


In [ ]:
# ═══ BLOQUE 3 · PARTIDAS HABILITADAS PARA EL PERIODO ═══
DRIVE_FOLDER_ID = "1gAO8-UjTV2bmBwMPgONBAYeVfpwXSWCY"   # carpeta CONTROL de Exportaciones

a1, a2 = sel_anio_ini.value, sel_anio_fin.value
if a1 > a2:
    raise ValueError("El año de inicio es posterior al año fin.")

def versiones(a1, a2):
    """Versiones del Arancel que rigen en algún mes de [ene a1, dic a2]."""
    fin = lambda i: (VIGOR[VERS[i + 1]] if i + 1 < len(VERS) else (9999, 12))
    return [v for i, v in enumerate(VERS) if VIGOR[v] <= (a2, 12) and fin(i) > (a1, 1)]

vs = versiones(a1, a2)
tramos = list(zip(vs, vs[1:]))
print(f"📅 {a1}–{a2} → versiones del Arancel: {', '.join(vs)}")

presente = dic[[f"V{v}" for v in vs]].eq("1")
continua = (dic[[f"C{a[2:]}_{b[2:]}" for a, b in tramos]].eq("1").all(axis=1)
            if tramos else pd.Series(True, index=dic.index))
habil = presente.all(axis=1) & continua
parcial = presente.any(axis=1) & ~habil

res = dic.assign(SITUACION="").copy()
res.loc[habil, "SITUACION"] = "HABILITADA"
res.loc[parcial, "SITUACION"] = "PARCIAL"
res["VERSIONES_EN_PERIODO"] = presente.apply(lambda r: " ".join(v for v, x in zip(vs, r) if x), axis=1)
res = res[res.SITUACION != ""]
if not chk_parciales.value:
    res = res[res.SITUACION == "HABILITADA"]
if sel_caps.value:
    res = res[res.CAPITULO.isin(sel_caps.value)]
q = txt_buscar.value.strip().lower()
if q:
    res = res[res.PARTIDA.str.startswith(q.replace(".", "")) |
              res.DESCRIPCION_COMPLETA.str.lower().str.contains(q, regex=False) |
              res.DESCRIPCION.str.lower().str.contains(q, regex=False)]

cols = ["SITUACION", "PARTIDA", "CODIGO_SUNAT", "CAPITULO", "CAPITULO_DESC", "PARTIDA_4", "PARTIDA_4_DESC",
        "DESCRIPCION", "VERSIONES_EN_PERIODO", "ESTADO", "SERIE_ESTABLE", "TRAMOS", "MISMA_DESCRIPCION",
        "EQUIV_ANTERIORES", "EQUIV_SIGUIENTES", "DESCRIPCION_COMPLETA"]
res = res[cols].sort_values(["CAPITULO", "PARTIDA"]).reset_index(drop=True)

total = dic[presente.any(axis=1)]
print(f"✅ HABILITADAS (serie comparable todo el periodo): {(res.SITUACION == 'HABILITADA').sum():,}")
if chk_parciales.value:
    print(f"🟡 PARCIALES (existen solo en parte del periodo; ver EQUIV_*): {(res.SITUACION == 'PARCIAL').sum():,}")
print(f"⛔ Deshabilitadas (no existen en {a1}–{a2}, en todo el Arancel): {len(dic) - len(total):,}")
display(res.groupby(["CAPITULO", "SITUACION"]).size().unstack(fill_value=0).head(100))
display(res.head(30))

if chk_drive.value:
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaFileUpload
    auth.authenticate_user()
    drive = build("drive", "v3", cache_discovery=False)
    caps = "TODOS" if not sel_caps.value else "-".join(sel_caps.value) if len(sel_caps.value) <= 5 else f"{len(sel_caps.value)}CAP"
    nombre = f"PARTIDAS - {a1}-{a2} - CAP {caps}.csv"
    ruta = f"/content/{nombre}"
    res.to_csv(ruta, index=False, encoding="utf-8-sig")
    r = drive.files().list(q=f"'{DRIVE_FOLDER_ID}' in parents and name = '{nombre}' and trashed = false",
                           fields="files(id)", supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
    media = MediaFileUpload(ruta, mimetype="text/csv", resumable=True)
    if r.get("files"):
        drive.files().update(fileId=r["files"][0]["id"], media_body=media, supportsAllDrives=True).execute()
    else:
        drive.files().create(body={"name": nombre, "parents": [DRIVE_FOLDER_ID]}, media_body=media,
                             fields="id", supportsAllDrives=True).execute()
    print(f"☁️  Guardado en Drive: {nombre}")
